# 📈 Market Sentiment Analyzer - Exploratory Data Analysis

This notebook explores the relationship between social media sentiment (Reddit) and stock price movements.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.storage import DatabaseManager, settings

# Setup
plt.style.use('ggplot')
db = DatabaseManager(settings.db_path)

## 1. Load Data

In [ ]:
ticker = "TSLA"
query = f"SELECT * FROM price_data WHERE ticker = '{ticker}' ORDER BY date"
prices = pd.read_sql(query, db.engine)
prices['date'] = pd.to_datetime(prices['date'])

query = f"SELECT * FROM reddit_daily_sentiment WHERE ticker = '{ticker}' ORDER BY date"
sentiment = pd.read_sql(query, db.engine)
sentiment['date'] = pd.to_datetime(sentiment['date'])

# Merge for analysis
df = pd.merge(prices, sentiment, on=['ticker', 'date'], how='inner')
df.head()

## 2. Sentiment-Price Correlation Analysis

In [ ]:
lags = [0, 1, 2, 3]
corrs = {}

for lag in lags:
    corrs[f'Lag {lag}'] = df['vader_compound'].corr(df['daily_return'].shift(-lag))

plt.figure(figsize=(10, 6))
pd.Series(corrs).plot(kind='bar')
plt.title(f'Sentiment-Price Correlation at Different Lags ({ticker})')
plt.ylabel('Correlation Coefficient')
plt.show()